Send Fraud Alert Email
Checks for newly flagged customer accounts (shipment/return round-trip pattern above threshold) and emails an alert only when something new appears.

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
import json
import io
import datetime
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery/analysis"

Load current flagged candidates and previous run's list

In [0]:
customer_fraud_summary = read_gold(blob_service, f"{ANALYSIS_BASE}/fraud_customer_summary.parquet")

flagged_candidates = customer_fraud_summary[
    (customer_fraud_summary["total_month_end_shipments"] >= fraud_min_shipments) &
    (customer_fraud_summary["round_trip_rate"] >= fraud_rate_threshold)
].sort_values("round_trip_rate", ascending=False)

try:
    blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_flagged_customers_previous.json")
    previous_flagged = set(json.loads(blob_client.download_blob().readall()))
except Exception:
    previous_flagged = set()

new_flags = set(flagged_candidates["customer_no_shipment"]) - previous_flagged
print(f"New flagged accounts since last run: {len(new_flags)}")
print(new_flags)

Only send if there's something new

In [0]:
if new_flags:
    new_flag_rows = flagged_candidates[flagged_candidates["customer_no_shipment"].isin(new_flags)]

    body = f"""Hello,

The automated fraud-pattern check has identified {len(new_flags)} new customer account(s) showing an elevated shipment/return round-trip rate near month-end (potential bonus-gaming pattern), above the {RATE_THRESHOLD:.0%} threshold.

NEWLY FLAGGED ACCOUNTS
"""
    for _, row in new_flag_rows.iterrows():
        body += (
            f"- {row['customer_name']} (salesperson: {row['primary_salesperson']}, "
            f"{row['distinct_salespeople']} distinct): "
            f"{row['round_trip_count']} round-trips of {row['total_month_end_shipments']} month-end shipments "
            f"({row['round_trip_rate']:.1%}), total value {row['total_value_involved']:,.2f}\n"
        )

    body += """
Please review the attached evidence file for transaction-level detail before taking any action. This is a pattern flag for investigation, not a confirmed finding.

This is an automated message from the Exide Sales Fraud Detection pipeline.
"""

    msg = MIMEMultipart()
    msg["From"] = smtp_username
    msg["To"] = ", ".join(fraud_alert)
    msg["Subject"] = f"⚠ Fraud Pattern Alert - {len(new_flags)} New Flagged Account(s) - {datetime.date.today().isoformat()}"
    msg.attach(MIMEText(body, "plain"))

    blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_flagged_evidence.xlsx")
    evidence_bytes = blob_client.download_blob().readall()
    attachment = MIMEApplication(evidence_bytes, _subtype="xlsx")
    attachment.add_header("Content-Disposition", "attachment", filename="fraud_flagged_evidence.xlsx")
    msg.attach(attachment)

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(smtp_username, smtp_password)
        server.sendmail(smtp_username, fraud_alert, msg.as_string())
        print(f"Alert email sent to {', '.join(fraud_alert)}")
    except Exception as e:
        print(f"Failed to send alert email: {e}")
    finally:
        server.quit()
else:
    print("No new flagged accounts - no email sent")

Update the "previous run" record regardless

In [0]:
all_flagged_ids = flagged_candidates["customer_no_shipment"].tolist()
blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_flagged_customers_previous.json")
blob_client.upload_blob(json.dumps(all_flagged_ids), overwrite=True)
print(f"Updated tracking list: {len(all_flagged_ids)} currently flagged accounts")